# 06 — Coupling-graph routing

Yen *k*-shortest paths from 5lpg **ligand** contacts to positions among the
lowest **ESM-2 650M vs −SPURS** deconvolution scores.

The graph is an EVcouplings `*CouplingScoresCompared_all.csv`
(shipped here; for a new protein download it from
[evcouplings.org](https://evcouplings.org)).


## Load previous outputs


In [ ]:
from IPython.display import display
import pandas as pd

from dualde.contacts import NUDT15_LIGAND_NAMES, attach_minimum_distance
from dualde.paths import example_dir
from dualde.routing import run_information_transfer_pipeline

EXAMPLE = example_dir("NUDT15_HUMAN")
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

deconv = pd.read_csv(OUT / "deconvolution.csv")
contacts = pd.read_csv(OUT / "5lpg_contacts.csv")
func = pd.read_csv(OUT / "5lpg_functional_residues.csv")
coup = pd.read_csv(EXAMPLE / "couplings" / "b0.5_CouplingScoresCompared_all.csv")

print("deconv", deconv.shape, "contacts", contacts.shape, "func", func.shape, "couplings", coup.shape)


Coupling table from the EVcouplings webserver (needs `i`, `j`, `probability`, `dist`):


In [ ]:
print("coupling columns:", list(coup.columns))
print(coup[["i", "j", "probability", "dist"]].describe().T)
coup.head()


In [ ]:
print("functional residues by type:")
print(func["contact_type"].value_counts())
print("ligand sources (residue_name in", list(NUDT15_LIGAND_NAMES), "):")
lig = func.loc[func["residue_name"].isin(NUDT15_LIGAND_NAMES)]
display(lig[["target_pos", "residue_name", "minimum_distance", "contact_type"]])


## Run routing


In [ ]:
scores = attach_minimum_distance(deconv, contacts)

candidates = [
    "ESM2_650M_neg_spurs_model",
    "EVmutation_neg_spurs_model",
]
score_col = next((c for c in candidates if c in scores.columns), None)
if score_col is None:
    raise ValueError("Run notebook 05 first (need ESM2_650M_neg_spurs_model).")
print("score_col:", score_col)

result = run_information_transfer_pipeline(
    all_merged_df=scores,
    couplings_file=str(EXAMPLE / "couplings" / "b0.5_CouplingScoresCompared_all.csv"),
    functional_residues_file=str(OUT / "5lpg_functional_residues.csv"),
    active_site_ligands=list(NUDT15_LIGAND_NAMES),
    activity_wt=0.45,
    abundance_wt=0.45,
    score_col=score_col,
    low_score_percentile=1.0,
    prob_threshold=0.7,
    n_paths=5,
    weighted=True,
    dist_cutoff=5,
    seq_len=164,
    verbose=True,
)

paths = result["paths_df"]
pairs = result["pairs_df"]
paths.to_csv(OUT / "coupling_paths.csv", index=False)
pairs.to_csv(OUT / "coupling_pairs.csv", index=False)
print("wrote", OUT / "coupling_paths.csv")


## Inspect paths


In [ ]:
print("n paths:", len(paths), "n pairs:", len(pairs))
print("pairs with a path:", int((pairs["n_paths_found"] > 0).sum()) if len(pairs) else 0)
print("sources:", sorted(paths["source"].unique()) if len(paths) else [])
print("targets:", sorted(paths["target"].unique()) if len(paths) else [])
paths.head(10)


In [ ]:
if len(paths):
    print("shortest paths (fewest edges):")
    display(paths.nsmallest(8, "n_edges")[["source", "target", "path_rank", "n_edges", "min_edge_prob", "path_str"]])
    print("pair status (first 12):")
    display(pairs.head(12))
else:
    print("No paths — loosen prob_threshold or check that notebook 01 found ligand contacts.")
